# Starter Kit FahMai RAG

This notebook walks through building a minimalist **Retrieval-Augmented Generation (RAG)** system for the FahMai electronics store Q&A challenge.

**Sections:**
- **Section 0:** Setup & LLM Test
- **Section 1:** Dense Retrieval (MiniLM)
- **Section 2:** Sparse Retrieval (BM25)
- **Section 3:** Hybrid Retrieval (RRF)
- **Section 4:** What's Next?

In [24]:
# === CONFIGURATION ===
# Change N_QUESTIONS to 100 for a full competition run.

N_QUESTIONS = 100  # 10 for demo, 100 for real submission
DATA_DIR = ''
KB_DIR = f"knowledge_base/"

---
## Section 0: Setup & LLM Test

First, install dependencies and test the ThaiLLM API — **without** any retrieval. This shows why RAG is needed.

ThaiLLM website: https://playground.thaillm.or.th/chat/

In [1]:
import os, csv, re, time, requests
import numpy as np
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

THAILLM_API_KEY = os.getenv("THAILLM_API_KEY")

In [ ]:
def ask_llm(messages, model="typhoon", max_retries=5):

    url = f"http://thaillm.or.th/api/{model}/v1/chat/completions"
    
    headers = {
        "Content-Type": "application/json",
        "apikey": THAILLM_API_KEY 
    }
    
    payload = {
        "model": "/model",
        "messages": messages,
        "max_tokens": 2048,
        "temperature": 0.0,
    }
  
    for attempt in range(max_retries):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=120)
            
            if response.status_code == 200:
                return response.json()["choices"][0]["message"]["content"].strip()
            
            elif response.status_code == 429:
                wait = min(2 ** attempt, 30)
                print(f"Rate limited, waiting {wait}s...")
                time.sleep(wait)
            else:
                print(f"Error {response.status_code}: {response.text}")
                if response.status_code in [401, 403]:
                    break
                    
        except Exception as e:
            print(f"Attempt {attempt+1} failed: {e}")
            time.sleep(2 ** attempt)

    return None

def parse_answer(text):
    if not text: return None
    clean = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()
    m = re.search(r"ANSWER:\s*(\d+)", clean)
    if m: return int(m.group(1))
    for d in re.findall(r"\b(\d{1,2})\b", clean):
        if 1 <= int(d) <= 10: return int(d)
    return None

### Test the LLM without RAG

Let's ask a FahMai-specific question *without* any context. The LLM shouldn't know the answer.

In [6]:
# Ask without context — LLM has no idea about FahMai's products
response = ask_llm([
    {"role": "user", "content": "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"}
])
print("LLM response (no context):", response)
print("\n→ The LLM doesn't know FahMai-specific facts. We need RAG!")

Error 401: {
  "message":"No API key found in request",
  "request_id":"759bdd1b00652e0ffcc73970e7ed655e"
}
LLM response (no context): None

→ The LLM doesn't know FahMai-specific facts. We need RAG!


### Load Questions

In [27]:
questions = []
with open(f"questions.csv", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        choices = {str(i): row[f"choice_{i}"] for i in range(1, 11)}
        questions.append({"id": int(row["id"]), "question": row["question"], "choices": choices})

print(f"Loaded {len(questions)} questions, using first {N_QUESTIONS} for this run")
print(f"\nExample — Q1: {questions[0]['question']}")
for k, v in questions[0]["choices"].items():
    print(f"  {k}. {v}")

Loaded 100 questions, using first 100 for this run

Example — Q1: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ
  1. 3 ATM
  2. IP68
  3. 5 ATM
  4. IP67
  5. 10 ATM
  6. 20 ATM
  7. IPX8
  8. 1 ATM
  9. ไม่มีข้อมูลนี้ในฐานข้อมูล
  10. คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า


In [5]:
questions

[{'id': 1,
  'question': 'Watch S3 Ultra กันน้ำได้กี่ ATM ครับ',
  'choices': {'1': '3 ATM',
   '2': 'IP68',
   '3': '5 ATM',
   '4': 'IP67',
   '5': '10 ATM',
   '6': '20 ATM',
   '7': 'IPX8',
   '8': '1 ATM',
   '9': 'ไม่มีข้อมูลนี้ในฐานข้อมูล',
   '10': 'คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า'}},
 {'id': 2,
  'question': 'ปากกา SaiFah Pen Gen 2 ราคาเท่าไหร่คะ',
  'choices': {'1': '2,990 บาท',
   '2': '4,490 บาท',
   '3': '1,990 บาท',
   '4': '4,990 บาท',
   '5': '3,490 บาท',
   '6': '2,490 บาท',
   '7': '3,990 บาท',
   '8': '5,490 บาท',
   '9': 'ไม่มีข้อมูลนี้ในฐานข้อมูล',
   '10': 'คำถามนี้ไม่เกี่ยวข้องกับฐานข้อมูลสินค้า'}},
 {'id': 3,
  'question': 'หูฟัง HeadPro X1 ใช้ Bluetooth เวอร์ชันอะไรคะ',
  'choices': {'1': 'Bluetooth 5.0',
   '2': 'Bluetooth 5.3',
   '3': 'Bluetooth 5.4',
   '4': 'Bluetooth 4.2',
   '5': 'Bluetooth 5.2',
   '6': 'Bluetooth 5.1',
   '7': 'Bluetooth 4.0',
   '8': 'Bluetooth 5.5',
   '9': 'ไม่มีข้อมูลนี้ในฐานข้อมูล',
   '10': 'คำถามนี้ไม่เกี่ยวข้องกับฐานข้อ

---
## Section 1: Dense Retrieval (MiniLM)

**Dense retrieval** converts text into vectors (embeddings) and finds relevant chunks by cosine similarity.

The pipeline: **Load docs → Chunk → Embed → Retrieve → Generate**

### 1.1 Load Knowledge Base

In [28]:
kb_dir = Path(KB_DIR)
documents = []
for fp in sorted(kb_dir.rglob("*.md")):
    text = fp.read_text(encoding="utf-8")
    documents.append({"path": str(fp.relative_to(kb_dir)), "text": text})

print(f"Loaded {len(documents)} documents")
print(f"  products/: {sum(1 for d in documents if 'products/' in d['path'])}")
print(f"  policies/: {sum(1 for d in documents if 'policies/' in d['path'])}")
print(f"  store_info/: {sum(1 for d in documents if 'store_info/' in d['path'])}")

# Preview one document
print(f"\n--- Sample: {documents[0]['path']} ---")
print(documents[0]["text"][:500])

Loaded 118 documents
  products/: 110
  policies/: 5
  store_info/: 3

--- Sample: policies/cancellation_policy.md ---
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้อเป็นหลัก

---

## 2. การยกเลิกตามสถานะคำสั่งซื้อ

### 2.1 สถานะ "รอชำระเงิน" (Pending Payment)

**ยกเลิกได้ทันที**

คำสั่งซื้อที่อยู่ในสถานะรอชำระเงินสามารถยกเลิกได้ทันทีโดยไม่มีค่าใช้จ่าย ผ่านแอปพลิ


### 1.2 Chunking

LLMs have limited context windows, and long documents dilute relevance. We split each document into smaller **chunks** using a fixed-size sliding window with overlap.

In [29]:
CHUNK_SIZE = 512
CHUNK_OVERLAP = 128

def make_chunks(text, size, overlap):
    """Split text into overlapping fixed-size windows."""
    if len(text) <= size:
        return [text]
    chunks = []
    start = 0
    while start < len(text):
        chunks.append(text[start : start + size])
        start += size - overlap
    return chunks

# Build all chunks
chunks = []
for doc in documents:
    for window in make_chunks(doc["text"], CHUNK_SIZE, CHUNK_OVERLAP):
        chunks.append({"text": window, "source": doc["path"]})

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\n--- Sample chunk ---")
print(f"Source: {chunks[0]['source']}")
print(chunks[0]["text"][:300])

Created 1055 chunks from 118 documents

--- Sample chunk ---
Source: policies/cancellation_policy.md
# นโยบายการยกเลิกคำสั่งซื้อ — ร้านฟ้าใหม่

**วันที่อัปเดต:** 1 มีนาคม 2569

---

## 1. ภาพรวมนโยบาย

ฟ้าใหม่เข้าใจว่าบางครั้งลูกค้าอาจต้องการยกเลิกคำสั่งซื้อด้วยเหตุผลต่างๆ นโยบายนี้อธิบายสิทธิ์และขั้นตอนการยกเลิกคำสั่งซื้อตามสถานะของคำสั่งซื้อในขณะนั้น ความสามารถในการยกเลิกขึ้นอยู่กับสถานะคำสั่งซื้


### 1.3 Embedding

We use `paraphrase-multilingual-MiniLM-L12-v2` — a small (118M params), multilingual embedding model that produces 384-dimensional vectors. It supports Thai out of the box and doesn't require any special prefixes.

In [30]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

# Embed all chunks
chunk_texts = [c["text"] for c in chunks]
chunk_embeddings = embed_model.encode(chunk_texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)

print(f"Chunk embeddings shape: {chunk_embeddings.shape}")  # (n_chunks, 384)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/17 [00:00<?, ?it/s]

Chunk embeddings shape: (1055, 384)


**BGE-M3 Rerank Chunk**

In [31]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('BAAI/bge-m3')

bge_chunk_embeddings = embed_model.encode(
    [c["text"] for c in chunks], 
    batch_size=32, 
    show_progress_bar=True, 
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

### 1.4 Retrieve

Embed the question, then find the most similar chunks via dot product (= cosine similarity for normalized vectors).

In [9]:
TOP_K = 5

def dense_retrieve(query, chunk_embs, k=TOP_K):
    """Return indices of top-k most similar chunks to the query."""
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embs, q_emb.T).flatten()  # cosine similarity (vectors are normalized)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Demo: retrieve for Q1
# q = questions[0]
# idx, scores = dense_retrieve(q["question"], chunk_embeddings)

# print(f"Question: {q['question']}\n")
# for rank, (i, s) in enumerate(zip(idx, scores), 1):
#     print(f"  Rank {rank} (score={s:.3f}) [{chunks[i]['source']}]")
#     print(f"  {chunks[i]['text'][:150]}...")
#     print()

### 1.5 Generate Answer

Send the retrieved context + question + choices to the LLM and parse the answer.

In [10]:
SYSTEM_PROMPT = """คุณคือผู้เชี่ยวชาญด้านข้อมูลผลิตภัณฑ์ร้าน FahMai Electronics
หน้าที่ของคุณคืออ่าน 'ข้อมูลอ้างอิง' แล้วตอบคำถามในรูปแบบตัวเลือกที่ถูกต้องที่สุด"""

def build_rag_prompt(question, choices, retrieved_chunks):
    context_block = "\n\n".join([f"--- ข้อมูลส่วนที่ {i+1} ---\n{c}" for i, c in enumerate(retrieved_chunks)])
    choices_block = "\n".join([f"{k}. {v}" for k, v in choices.items()])
    return f"""ใช้ข้อมูลอ้างอิงด้านล่างเพื่อตอบคำถามในรูปแบบตัวเลือก

    ### ข้อมูลอ้างอิง:
    {context_block}

    ### คำถาม:
    {question}

    ### ตัวเลือก:
    {choices_block}

    ---
    ### ขั้นตอนการทำงาน:
    1. พิจารณาว่าคำถามถามถึงอะไร และข้อมูลอ้างอิงส่วนไหนที่เกี่ยวข้อง
    2. วิเคราะห์ตัวเลือกที่ถูกต้องโดยเปรียบเทียบกับข้อมูลอ้างอิง
    3. ระบุเหตุผลสั้นๆ แล้วตอบในรูปแบบ 'ANSWER: X' (X คือเลขตัวเลือก 1-10) เท่านั้น

    วิเคราะห์และตอบ:"""

# Demo: answer Q1
# q = questions[0]
# idx, _ = dense_retrieve(q["question"], chunk_embeddings)
# retrieved = [chunks[i] for i in idx]

# prompt = build_rag_prompt(q["question"], q["choices"], retrieved)
# raw = ask_llm([
# {"role": "system", "content": SYSTEM_PROMPT},
# {"role": "user", "content": prompt},
# ])
# answer = parse_answer(raw)
# print(f"Q{q['id']}: {q['question']}")
# print(f"LLM raw: {raw}")
# print(f"Parsed answer: {answer}")

### 1.6 Run All Questions (Dense)

Loop through questions, retrieve, generate, and collect predictions.

In [11]:
def run_pipeline(questions, retrieve_fn, label="dense", n=N_QUESTIONS):
    """Run retrieval + LLM on first n questions. Returns predictions dict."""
    predictions = {}
    for i, q in enumerate(questions[:n]):
        retrieved_chunks = retrieve_fn(q["question"])
        prompt = build_rag_prompt(q["question"], q["choices"], retrieved_chunks)
        raw = ask_llm([
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": prompt},
        ])
        pred = parse_answer(raw)
        predictions[q["id"]] = pred if pred else 1  # default to 1 if parse fails
        print(f"  Q{q['id']:>3}: pred={predictions[q['id']]}")
        time.sleep(0.3)  # be nice to the API
    print(f"\n{label}: answered {len(predictions)} questions")
    return predictions

# Dense retrieval function
def dense_retrieve_chunks(query):
    idx, _ = dense_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

# dense_preds = run_pipeline(questions, dense_retrieve_chunks, label="dense")

---
## Section 2: Sparse Retrieval (BM25)

**BM25** is a keyword-matching algorithm. It scores documents by how many query terms they contain, weighted by term rarity (IDF). No neural network needed.

### 2.1 Thai Tokenization

BM25 needs tokenized text. Thai has no spaces between words, so we use `pythainlp` to segment.

In [32]:
from pythainlp.tokenize import word_tokenize

# Demo: tokenize a Thai sentence
sample = "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"
tokens = word_tokenize(sample, engine="newmm")
print(f"Input:  {sample}")
print(f"Tokens: {tokens}")

Input:  Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?
Tokens: ['Watch', ' ', 'S', '3', ' ', 'Ultra', ' ', 'กันน้ำ', 'ได้', 'กี่', ' ', 'ATM', ' ', 'ครับ', '?']


### 2.2 Build BM25 Index

In [33]:
from rank_bm25 import BM25Okapi

# Tokenize all chunks
tokenized_chunks = [word_tokenize(c["text"], engine="newmm") for c in chunks]
bm25 = BM25Okapi(tokenized_chunks)

print(f"BM25 index built over {len(tokenized_chunks)} chunks")

BM25 index built over 1055 chunks


### 2.3 Retrieve with BM25

Compare BM25 results with dense results for the same question.

In [34]:
def bm25_retrieve(query, k=TOP_K):
    """Return top-k chunk indices using BM25."""
    tokens = word_tokenize(query, engine="newmm")
    scores = bm25.get_scores(tokens)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Compare: same question, two retrieval methods
q = questions[0]
print(f"Question: {q['question']}\n")

d_idx, _ = dense_retrieve(q["question"], chunk_embeddings)
b_idx, _ = bm25_retrieve(q["question"])

print("Dense top-5 sources:")
for i in d_idx:
    print(f"  {chunks[i]['source']}")

print("\nBM25 top-5 sources:")
for i in b_idx:
    print(f"  {chunks[i]['source']}")

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ



ValueError: shapes (1055,384) and (1024,1) not aligned: 384 (dim 1) != 1024 (dim 0)

### 2.4 Run All Questions (BM25)

In [38]:
def bm25_retrieve_chunks(query):
    idx, _ = bm25_retrieve(query)
    return [chunks[i] for i in idx]

bm25_preds = run_pipeline(questions, bm25_retrieve_chunks, label="bm25")

  Q  1: pred=5
  Q  2: pred=7
  Q  3: pred=9
  Q  4: pred=6
  Q  5: pred=6
  Q  6: pred=9
  Q  7: pred=9
  Q  8: pred=1
  Q  9: pred=2
  Q 10: pred=2
  Q 11: pred=4
  Q 12: pred=1
  Q 13: pred=6
  Q 14: pred=4
  Q 15: pred=7
  Q 16: pred=1
  Q 17: pred=8
  Q 18: pred=5
  Q 19: pred=2
  Q 20: pred=2
  Q 21: pred=3
  Q 22: pred=6
  Q 23: pred=3
  Q 24: pred=3
  Q 25: pred=5
  Q 26: pred=6
  Q 27: pred=2
  Q 28: pred=7
  Q 29: pred=4
  Q 30: pred=3
  Q 31: pred=4
Error 502: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>thaillm.or.th | 502: Bad gateway</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<m

---
## Section 3: Hybrid Retrieval (RRF)

**Hybrid** combines dense and sparse results. The idea: dense is good at semantic matching, BM25 is good at exact keyword matching. Together they cover more cases.

We use **Reciprocal Rank Fusion (RRF)** to merge the two ranked lists:

$$\text{score}(d) = \sum_{r \in \text{rankers}} \frac{1}{k + \text{rank}_r(d)}$$

where $k$ is a constant (typically 60). Documents ranked highly by *either* method get a high combined score.

In [21]:
def hybrid_retrieve(query, chunk_embs, k=TOP_K, rrf_k=60):
    """Combine dense + BM25 results using Reciprocal Rank Fusion."""
    # Get top candidates from each method (fetch more than k to improve fusion)
    fetch_k = k * 2
    d_idx, _ = dense_retrieve(query, chunk_embs, k=fetch_k)
    b_idx, _ = bm25_retrieve(query, k=fetch_k)

    # Compute RRF scores
    rrf_scores = {}
    for rank, idx in enumerate(d_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(b_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)

    # Sort by combined score, return top-k
    sorted_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:k]
    return sorted_idx

# Demo
q = questions[0]
h_idx = hybrid_retrieve(q["question"], chunk_embeddings)
print(f"Question: {q['question']}\n")
print("Hybrid top-5 sources:")
for i in h_idx:
    print(f"  {chunks[i]['source']}")

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

Hybrid top-5 sources:
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-003_wongkhojon_watch_s3.md
  products/WK-SW-001_wongkhojon_watch_s3_ultra.md
  products/WK-SW-002_wongkhojon_watch_s3_pro.md
  products/WK-SW-003_wongkhojon_watch_s3.md


### 3.2 Run All Questions (Hybrid)

In [22]:
def hybrid_retrieve_chunks(query):
    idx = hybrid_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

hybrid_preds = run_pipeline(questions, hybrid_retrieve_chunks, label="hybrid")

  Q  1: pred=5
  Q  2: pred=7
  Q  3: pred=9
  Q  4: pred=6
  Q  5: pred=6
  Q  6: pred=8
  Q  7: pred=9
  Q  8: pred=4
  Q  9: pred=9
  Q 10: pred=2
  Q 11: pred=4
  Q 12: pred=1
  Q 13: pred=6
  Q 14: pred=1
  Q 15: pred=7
  Q 16: pred=1
  Q 17: pred=8
  Q 18: pred=5
  Q 19: pred=2
Error 502: <!DOCTYPE html>
<!--[if lt IE 7]> <html class="no-js ie6 oldie" lang="en-US"> <![endif]-->
<!--[if IE 7]>    <html class="no-js ie7 oldie" lang="en-US"> <![endif]-->
<!--[if IE 8]>    <html class="no-js ie8 oldie" lang="en-US"> <![endif]-->
<!--[if gt IE 8]><!--> <html class="no-js" lang="en-US"> <!--<![endif]-->
<head>

<title>thaillm.or.th | 502: Bad gateway</title>
<meta charset="UTF-8" />
<meta http-equiv="Content-Type" content="text/html; charset=UTF-8" />
<meta http-equiv="X-UA-Compatible" content="IE=Edge" />
<meta name="robots" content="noindex, nofollow" />
<meta name="viewport" content="width=device-width,initial-scale=1" />
<link rel="stylesheet" id="cf_styles-css" href="/cdn-cgi/styl

### BGE M3

In [35]:
from pythainlp.tokenize import word_tokenize

# Demo: tokenize a Thai sentence
sample = "Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?"
tokens = word_tokenize(sample, engine="newmm")
print(f"Input:  {sample}")
print(f"Tokens: {tokens}")

Input:  Watch S3 Ultra กันน้ำได้กี่ ATM ครับ?
Tokens: ['Watch', ' ', 'S', '3', ' ', 'Ultra', ' ', 'กันน้ำ', 'ได้', 'กี่', ' ', 'ATM', ' ', 'ครับ', '?']


In [ ]:
from sentence_transformers import SentenceTransformer

embed_model = SentenceTransformer('BAAI/bge-m3')


print("Encoding chunks with BGE-m3... (This might take a while)")
bge_chunk_embeddings = embed_model.encode(
    [c["text"] for c in chunks], 
    batch_size=32, 
    show_progress_bar=True, 
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Encoding chunks with BGE-m3... (This might take a while)


Batches:   0%|          | 0/33 [00:00<?, ?it/s]

In [ ]:
TOP_K = 5

def BGE_retrieve(query, chunk_embs, k=TOP_K):
    """Return indices of top-k most similar chunks to the query."""
    q_emb = embed_model.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embs, q_emb.T).flatten()  # cosine similarity (vectors are normalized)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Demo: retrieve for Q1
q = questions[0]
idx, scores = BGE_retrieve(q["question"], bge_chunk_embeddings)

print(f"Question: {q['question']}\n")
for rank, (i, s) in enumerate(zip(idx, scores), 1):
    print(f"  Rank {rank} (score={s:.3f}) [{chunks[i]['source']}]")
    print(f"  {chunks[i]['text'][:150]}...")
    print()

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

  Rank 1 (score=0.735) [products/WK-SW-002_wongkhojon_watch_s3_pro.md]
  3 Pro มีฟังก์ชัน ECG (คลื่นไฟฟ้าหัวใจ) ครบ นี่คือความแตกต่างหลักจาก Watch S3 (WK-SW-003) ที่ไม่มี ECG ส่วนฟีเจอร์อื่นอย่าง SpO2, GPS, AMOLED, และกันน้...

  Rank 2 (score=0.732) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
  ือน** ผ่านบัตรเครดิตที่ร่วมรายการ
(หมดเขต 30 มิถุนายน 2569)

ตัวอย่าง: ฿14,990 ÷ 6 เดือน = เพียง ฿2,498.33/เดือน

---

## คำถามที่พบบ่อยเกี่ยวกับสินค้...

  Rank 3 (score=0.722) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
  กาศ (Grade 5 Titanium) น้ำหนักเบากว่าสแตนเลสทั่วไปถึง 40% แต่แข็งแกร่งกว่าหลายเท่า ให้ความรู้สึกสวมใส่ที่เบาสบายตลอดทั้งวันแม้ระหว่างออกกำลังกายหนัก

...

  Rank 4 (score=0.712) [products/WK-SW-001_wongkhojon_watch_s3_ultra.md]
  ve Mode, (3) GPS แบบ Dual-Band แม่นยำกว่า S3 Pro ที่เป็น Single-Band, (4) หน้าจอใหญ่กว่า 1.9 นิ้ว vs 1.7 นิ้ว, (5) แบตเตอรี่อึดกว่า ทั้งสองรุ่นมี ECG ...

  Rank 5 (score=0.692) [products/W

In [21]:
def BGE_retrieve_chunks(query):
    idx, _ = BGE_retrieve(query, chunk_embeddings)
    return [chunks[i] for i in idx]

bge_preds = run_pipeline(questions, BGE_retrieve_chunks, label="bge")

  Q  1: pred=5
  Q  2: pred=7
  Q  3: pred=2
  Q  4: pred=6
  Q  5: pred=6
  Q  6: pred=8
  Q  7: pred=1
  Q  8: pred=4
  Q  9: pred=4
  Q 10: pred=2
  Q 11: pred=4
  Q 12: pred=1
  Q 13: pred=6
  Q 14: pred=3
  Q 15: pred=7
  Q 16: pred=1
  Q 17: pred=8
  Q 18: pred=5
  Q 19: pred=2
  Q 20: pred=2
  Q 21: pred=3
  Q 22: pred=3
  Q 23: pred=3
  Q 24: pred=3
  Q 25: pred=5
  Q 26: pred=6
  Q 27: pred=2
  Q 28: pred=7
  Q 29: pred=4
  Q 30: pred=3
  Q 31: pred=4
  Q 32: pred=2
  Q 33: pred=8
  Q 34: pred=5
  Q 35: pred=3
  Q 36: pred=2
  Q 37: pred=8
  Q 38: pred=6
  Q 39: pred=4
  Q 40: pred=8
  Q 41: pred=7
  Q 42: pred=2
  Q 43: pred=4
  Q 44: pred=1
  Q 45: pred=2
  Q 46: pred=1
  Q 47: pred=2
  Q 48: pred=2
  Q 49: pred=6
  Q 50: pred=5
  Q 51: pred=7
  Q 52: pred=4
  Q 53: pred=9
  Q 54: pred=9
  Q 55: pred=9
  Q 56: pred=9
  Q 57: pred=9
  Q 58: pred=9
  Q 59: pred=9
  Q 60: pred=9
  Q 61: pred=9
  Q 62: pred=10
  Q 63: pred=9
  Q 64: pred=5
  Q 65: pred=3
  Q 66: pred=7
  Q 67: p

### BGE-M3-THAI

In [12]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel('jaeyong2/bge-m3-Thai',  use_fp16=True) 

chunks_text = [c["text"] for c in chunks]

print("Encoding chunks with BGE-m3-Thai...")
thai_chunk_embeddings = model.encode(
    chunks_text, 
    batch_size=32, 
)['dense_vecs']

print("Encoding complete!")

KeyboardInterrupt: 

In [13]:
from sentence_transformers import SentenceTransformer, models

word_embedding_model = models.Transformer('jaeyong2/bge-m3-Thai')

pooling_model = models.Pooling(word_embedding_model.get_word_embedding_dimension(), pooling_mode='cls')

model = SentenceTransformer(modules=[word_embedding_model, pooling_model])

# ใช้งานตามปกติ
thai_chunk_embeddings = model.encode(
    [c["text"] for c in chunks],
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True
)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/33 [00:00<?, ?it/s]

In [14]:
TOP_K = 5

def BGE_retrieve_thai(query, chunk_embs, k=TOP_K):
    """Return indices of top-k most similar chunks to the query."""
    q_emb = model.encode([query], normalize_embeddings=True)
    scores = np.dot(chunk_embs, q_emb.T).flatten()  # cosine similarity (vectors are normalized)
    top_idx = np.argsort(scores)[::-1][:k]
    return top_idx, scores[top_idx]

# Demo: retrieve for Q1
q = questions[0]
idx, scores = BGE_retrieve_thai(q["question"], thai_chunk_embeddings)

print(f"Question: {q['question']}\n")
for rank, (i, s) in enumerate(zip(idx, scores), 1):
    print(f"  Rank {rank} (score={s:.3f}) [{chunks[i]['source']}]")
    print(f"  {chunks[i]['text'][:150]}...")
    print()

Question: Watch S3 Ultra กันน้ำได้กี่ ATM ครับ

  Rank 1 (score=0.569) [products/SF-SP-004_saifah_phone_v7.md]
  เสียหายจากน้ำไม่ครอบคลุมการรับประกันไม่ว่ากรณีใด
...

  Rank 2 (score=0.374) [products/WK-SW-002_wongkhojon_watch_s3_pro.md]
   เมตร (5 ATM) — รองรับว่ายน้ำ, ไม่รองรับ Dive Mode |
| NFC | รองรับ (NFC Pay) |
| แบตเตอรี่ | สูงสุด 48 ชั่วโมง (โหมดปกติ) / 14 วัน (โหมดประหยัด) |
| ...

  Rank 3 (score=0.374) [products/WK-SW-003_wongkhojon_watch_s3.md]
  ATM) — รองรับว่ายน้ำ |
| แบตเตอรี่ | สูงสุด 18 ชั่วโมง (โหมดปกติ) / 10 วัน (โหมดประหยัด) |
| ชาร์จ | Magnetic Wireless Charging |
| ระบบปฏิบัติการ | W...

  Rank 4 (score=0.353) [products/SF-SP-005_saifah_phone_v7_lite.md]
   Points จะถูกเพิ่มภายใน 7 วันหลังได้รับสินค้า
...

  Rank 5 (score=0.342) [products/WK-SW-003_wongkhojon_watch_s3.md]
  ATM ในราคาที่เข้าถึงได้ง่ายกว่า S3 Pro กว่าครึ่ง

ตัวเรือนอะลูมิเนียมอัลลอยด์เคลือบผิวแบบ anodized ให้ความทนทานและน้ำหนักเบาพร้อมกัน หน้าจอ AMOLED 1.7...



In [15]:
def BGE_retrieve_thai_chunks(query):
    idx, _ = BGE_retrieve_thai(query, thai_chunk_embeddings)
    return [chunks[i] for i in idx]

bge_thai_preds = run_pipeline(questions, BGE_retrieve_thai_chunks, label="bge-thai")

  Q  1: pred=3
  Q  2: pred=7
  Q  3: pred=9
  Q  4: pred=6
  Q  5: pred=9
  Q  6: pred=9
  Q  7: pred=9
  Q  8: pred=4
  Q  9: pred=2
  Q 10: pred=3
  Q 11: pred=4
  Q 12: pred=9
  Q 13: pred=2
  Q 14: pred=9
  Q 15: pred=7
  Q 16: pred=1
  Q 17: pred=8
  Q 18: pred=5
  Q 19: pred=2
  Q 20: pred=2
  Q 21: pred=3
  Q 22: pred=6
  Q 23: pred=3
  Q 24: pred=3
  Q 25: pred=5
  Q 26: pred=6
  Q 27: pred=7
  Q 28: pred=7
  Q 29: pred=4
  Q 30: pred=3
  Q 31: pred=4
  Q 32: pred=2
  Q 33: pred=8
  Q 34: pred=5
  Q 35: pred=3
  Q 36: pred=2
  Q 37: pred=8
  Q 38: pred=6
  Q 39: pred=4
  Q 40: pred=9
  Q 41: pred=9
  Q 42: pred=2
  Q 43: pred=9
  Q 44: pred=9
  Q 45: pred=2
  Q 46: pred=1
  Q 47: pred=1
  Q 48: pred=2
  Q 49: pred=3
  Q 50: pred=5
  Q 51: pred=7
  Q 52: pred=1
  Q 53: pred=9
  Q 54: pred=9
  Q 55: pred=10
  Q 56: pred=9
  Q 57: pred=9
  Q 58: pred=9
  Q 59: pred=9
  Q 60: pred=10
  Q 61: pred=10
  Q 62: pred=10
  Q 63: pred=9
  Q 64: pred=5
  Q 65: pred=3
  Q 66: pred=7
  Q 67

### Hybrid BM25 & BGE

In [ ]:
def hybrid_retrieve_2(query, chunk_embs, k=TOP_K, rrf_k=60):
    """Combine dense + BM25 results using Reciprocal Rank Fusion."""
    # Get top candidates from each method (fetch more than k to improve fusion)
    fetch_k = k * 2
    d_idx, _ = BGE_retrieve(query, chunk_embs, k=fetch_k)
    b_idx, _ = bm25_retrieve(query, k=fetch_k)

    # Compute RRF scores
    rrf_scores = {}
    for rank, idx in enumerate(d_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)
    for rank, idx in enumerate(b_idx, 1):
        rrf_scores[idx] = rrf_scores.get(idx, 0) + 1.0 / (rrf_k + rank)

    # Sort by combined score, return top-k
    sorted_idx = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:k]
    return sorted_idx

# Demo
q = questions[0]
h_idx = hybrid_retrieve_2(q["question"], chunk_embeddings)
print(f"Question: {q['question']}\n")
print("Hybrid top-5 sources:")
for i in h_idx:
    print(f"  {chunks[i]['source']}")

In [ ]:
def hybrid_retrieve_chunks_2(query):
    idx = hybrid_retrieve_2(query, bge_chunk_embeddings)
    return [chunks[i] for i in idx]

hybrid_preds = run_pipeline(questions, hybrid_retrieve_chunks_2, label="hybrid")

### 3.3 Compare All Three Methods

In [43]:
print(f"{'QID':>4}  {'Dense':>6} {'BM25':>6} {'Hybrid':>7} {'BGE':>6}")
print("-" * 45)
for q in questions[:N_QUESTIONS]:
    qid = q["id"]
    d = dense_preds.get(qid, "-")
    b = bm25_preds.get(qid, "-")
    h = hybrid_preds.get(qid, "-")
    g = bge_preds.get(qid, "-")
    match = "" if d == b == h == g else "  ← differ"
    print(f"Q{qid:>3}  {d:>6} {b:>6} {h:>7} {g:>6} {match}")

 QID   Dense   BM25  Hybrid    BGE
---------------------------------------------
Q  1       5      5       5      5 
Q  2       7      7       7      7 
Q  3       2      9       9      2   ← differ
Q  4       6      6       6      6 
Q  5       6      6       6      6 
Q  6       8      9       8      8   ← differ
Q  7       1      9       9      1   ← differ
Q  8       4      1       4      4   ← differ
Q  9       2      2       9      4   ← differ
Q 10       7      2       2      2   ← differ
Q 11       4      4       4      4 
Q 12       9      1       1      1   ← differ
Q 13       3      6       6      6   ← differ
Q 14       3      4       1      2   ← differ
Q 15       7      7       7      7 
Q 16       1      1       1      1 
Q 17       8      8       8      8 
Q 18       5      5       5      5 
Q 19       2      2       2      2 
Q 20       2      2       2      2 
Q 21       3      3       7      3   ← differ
Q 22       6      6       6      6 
Q 23       1      3       1

### Write Submission

Pick your best method and generate a `submission.csv` for Kaggle.

> Set `N_QUESTIONS = 100` at the top and re-run the notebook to generate a full submission.

In [24]:
bge_thai_preds

{1: 9,
 2: 7,
 3: 9,
 4: 6,
 5: 9,
 6: 9,
 7: 10,
 8: 9,
 9: 9,
 10: 7,
 11: 4,
 12: 9,
 13: 2,
 14: 9,
 15: 7,
 16: 1,
 17: 8,
 18: 5,
 19: 1,
 20: 9,
 21: 7,
 22: 5,
 23: 4,
 24: 6,
 25: 5,
 26: 6,
 27: 7,
 28: 7,
 29: 4,
 30: 3,
 31: 4,
 32: 2,
 33: 8,
 34: 5,
 35: 3,
 36: 2,
 37: 8,
 38: 9,
 39: 4,
 40: 9,
 41: 9,
 42: 2,
 43: 9,
 44: 9,
 45: 2,
 46: 1,
 47: 5,
 48: 2,
 49: 9,
 50: 5,
 51: 7,
 52: 1,
 53: 9,
 54: 9,
 55: 9,
 56: 9,
 57: 9,
 58: 9,
 59: 9,
 60: 10,
 61: 10,
 62: 10,
 63: 9,
 64: 5,
 65: 3,
 66: 7,
 67: 6,
 68: 9,
 69: 2,
 70: 8,
 71: 4,
 72: 7,
 73: 6,
 74: 5,
 75: 6,
 76: 9,
 77: 3,
 78: 7,
 79: 5,
 80: 2,
 81: 6,
 82: 9,
 83: 3,
 84: 9,
 85: 2,
 86: 9,
 87: 1,
 88: 7,
 89: 4,
 90: 9,
 91: 4,
 92: 4,
 93: 5,
 94: 1,
 95: 1,
 96: 3,
 97: 7,
 98: 3,
 99: 4,
 100: 1}

In [22]:
# Use dense predictions as the submission (change to hybrid_preds or bm25_preds to try others)
best_preds = bge_preds

with open("submission_BGE_4.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["id", "answer"])
    for q in questions:
        qid = q["id"]
        writer.writerow([qid, best_preds.get(qid, 1)])  # default=1 for unanswered

print(f"submission.csv written ({len(questions)} rows)")

submission.csv written (100 rows)


---
## Section 4: What's Next?

This baseline uses a simple setup. Here are ideas to improve your score:

- **Better embeddings** — try other, stronger multilingual  embedding.
- **Smarter chunking** — split by structure or other methods or add useful information to each chunk
- **Chunk size tuning** — experiment with  256, 512, 1024 or something else character chunks
- **Different ThaiLLMs** — try `openthaigpt`, `kbtg`, `pathumma`.
- **Prompt engineering** — adjust the system prompt, add few-shot examples, or change the output format
- **Reranking** — use a cross-encoder or specialized reranker to re-score the top-k chunks before sending to the

**Feel free to implement your own RAG.**